# Dashboard — Analyse des tables Gold

Ce notebook propose une **visualisation alternative** des tables Gold du DataLake Météo :
<code>daily_aggregates</code>, <code>weekly_trends</code> et <code>extreme_events</code>,
lues directement depuis HDFS via l'API **WebHDFS** (REST).

Contrairement au dashboard Streamlit (<code>dashboard/app.py</code>), tout est affiché ici
avec <code>matplotlib</code> / <code>seaborn</code>.


In [ ]:
# Imports & helpers WebHDFS
import os
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

HDFS_NAMENODE = os.environ.get("HDFS_NAMENODE", "namenode")
HDFS_WEBHDFS_PORT = os.environ.get("HDFS_WEBHDFS_PORT", "9870")
BASE = "http://{0}:{1}/webhdfs/v1".format(HDFS_NAMENODE, HDFS_WEBHDFS_PORT)


def webhdfs_list(remote_dir):
    """Liste les entrées d'un répertoire HDFS via WebHDFS (op=LISTSTATUS)."""
    url = BASE + remote_dir
    resp = requests.get(url, params={"op": "LISTSTATUS", "user.name": "root"}, timeout=20)
    if resp.status_code == 404:
        return []
    resp.raise_for_status()
    data = resp.json()
    return [entry["pathSuffix"] for entry in data["FileStatuses"]["FileStatus"]]


def download_file(remote_path, local_path):
    """Télécharge un fichier HDFS vers le disque local (op=OPEN)."""
    url = BASE + remote_path
    with requests.get(url, params={"op": "OPEN", "user.name": "root"},
                      timeout=30, stream=True) as resp:
        resp.raise_for_status()
        with open(local_path, "wb") as fh:
            for chunk in resp.iter_content(chunk_size=1048576):
                if chunk:
                    fh.write(chunk)


def read_gold_table(table, max_files=200):
    """Télécharge et lit tous les fichiers *.parquet d'une table Gold."""
    remote_dir = "/gold/meteo/" + table
    stack = [remote_dir]
    parquet_files = []
    while stack:
        current = stack.pop()
        try:
            entries = webhdfs_list(current)
        except requests.RequestException:
            continue
        for name in entries:
            if name.startswith("_") or name.startswith("."):
                continue
            full = current + "/" + name
            if name.endswith(".parquet"):
                parquet_files.append(full)
            else:
                stack.append(full)

    frames = []
    tmpdir = tempfile.mkdtemp(prefix="meteo_")
    for i, remote in enumerate(parquet_files[:max_files]):
        local = os.path.join(tmpdir, "part_" + str(i) + ".parquet")
        try:
            download_file(remote, local)
            frames.append(pd.read_parquet(local))
        except Exception:
            continue
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


daily = read_gold_table("daily_aggregates")
weekly = read_gold_table("weekly_trends")
if not daily.empty and "dt" in daily.columns:
    daily["dt"] = pd.to_datetime(daily["dt"], errors="coerce")
print("daily_aggregates :", daily.shape)
print("weekly_trends    :", weekly.shape)


In [ ]:
# KPI texte : dernière date, températures moyennes, records
if daily.empty or "dt" not in daily.columns:
    print("Aucune donnée daily_aggregates disponible pour le moment.")
else:
    daily = daily.dropna(subset=["dt"])
    if daily.empty:
        print("Aucune donnée temporelle valide.")
    else:
        last_dt = daily["dt"].max()
        print("Dernière date disponible :", last_dt.date())
        print("Températures moyennes par ville (dernier jour) :")
        last_day = daily[daily["dt"] == last_dt]
        for city, grp in last_day.groupby("city"):
            temp = grp["temp_avg"].dropna()
            if not temp.empty:
                print("  - {0:10s} : {1:.1f} °C".format(city, temp.mean()))
        print("Records historiques :")
        for col in ("temp_max", "temp_min"):
            if col not in daily.columns:
                continue
            idx = daily[col].idxmax() if col == "temp_max" else daily[col].idxmin()
            row = daily.loc[idx]
            print("  - {0} : {1:.1f} °C à {2} le {3}".format(
                col, row[col], row["city"], row["dt"].date()))


In [ ]:
# Évolution temp_avg par ville (source OPENMETEO) sur 30 jours
if daily.empty or "dt" not in daily.columns:
    print("Pas de données à tracer.")
else:
    last_dt = daily["dt"].max()
    mask = daily["dt"] >= (last_dt - pd.Timedelta(days=30))
    openmeteo = daily[mask]
    if "source" in openmeteo.columns:
        openmeteo = openmeteo[openmeteo["source"].astype(str).str.upper() == "OPENMETEO"]
    plt.figure(figsize=(12, 6))
    for city, grp in openmeteo.sort_values("dt").groupby("city"):
        plt.plot(grp["dt"], grp["temp_avg"], marker="o", label=city)
    plt.title("Évolution de la température moyenne (30 jours, Open-Meteo)")
    plt.xlabel("Date")
    plt.ylabel("Température moyenne (°C)")
    plt.legend(title="Ville")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Bar plot hebdomadaire : précipitations cumulées (ou écart vs semaine précédente)
if weekly.empty:
    print("Pas de données weekly_trends à tracer.")
else:
    weekly = weekly.copy()
    weekly["week_label"] = weekly["year"].astype(str) + "-S" + weekly["week"].astype(str).str.zfill(2)
    weekly = weekly.sort_values(["year", "week"])
    if "precip_sum" in weekly.columns:
        pivot = weekly.pivot_table(index="week_label", columns="city",
                                   values="precip_sum", aggfunc="sum")
        ylabel = "Précipitations (mm)"
        titre = "Précipitations cumulées par semaine et par ville"
    else:
        pivot = weekly.pivot_table(index="week_label", columns="city",
                                   values="temp_vs_prev_week", aggfunc="mean")
        ylabel = "Écart vs semaine précédente (°C)"
        titre = "Écart de température vs semaine précédente"
    pivot.plot(kind="bar", figsize=(12, 5), alpha=0.85)
    plt.title(titre)
    plt.xlabel("Semaine")
    plt.ylabel(ylabel)
    plt.xticks(rotation=45)
    plt.legend(title="Ville", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
# Countplot des événements extrêmes par type (si extreme_events disponible)
try:
    events = read_gold_table("extreme_events")
except Exception:
    events = pd.DataFrame()
if events.empty or "event_type" not in events.columns:
    print("Pas de données extreme_events disponibles.")
else:
    plt.figure(figsize=(10, 5))
    order = events["event_type"].value_counts().index
    sns.countplot(data=events, x="event_type", order=order)
    plt.title("Nombre d'événements extrêmes par type")
    plt.xlabel("Type d'événement")
    plt.ylabel("Nombre d'occurrences")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Heatmap : température moyenne mensuelle par ville
if daily.empty or "dt" not in daily.columns or "temp_avg" not in daily.columns:
    print("Pas assez de données pour la heatmap.")
else:
    daily = daily.copy()
    daily["month"] = daily["dt"].dt.to_period("M").astype(str)
    pivot = daily.pivot_table(index="month", columns="city", values="temp_avg", aggfunc="mean")
    if pivot.empty or pivot.shape[0] < 2:
        print("Pas assez de données mensuelles pour la heatmap.")
    else:
        plt.figure(figsize=(10, 6))
        sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdBu_r",
                    cbar_kws={"label": "Température moyenne (°C)"})
        plt.title("Température moyenne mensuelle par ville (heatmap)")
        plt.xlabel("Ville")
        plt.ylabel("Mois")
        plt.tight_layout()
        plt.show()


## Conclusion

Ce notebook confirme que les tables Gold (<code>daily_aggregates</code>, <code>weekly_trends</code>,
<code>extreme_events</code>) sont directement exploitables pour la visualisation :

1. les agrégats journaliers permettent de suivre l'évolution des températures par ville ;
2. les tendances hebdomadaires mettent en évidence les écarts de température d'une semaine à l'autre ;
3. les événements extrêmes alimentent un suivi des alertes (chaleur, froid, vent, précipitations).

Le dashboard Streamlit (<code>dashboard/app.py</code>) reprend ces mêmes indicateurs dans une interface interactive.
